# Retrieval and evidence EDA

This notebook is a read-only diagnostic companion for the persisted traces produced by `uv run -m uni_rag_agent evidence build`. It never scans or executes files under `Courses`, writes SQLite, mutates Chroma, or rewrites packet JSON. The database connection below uses SQLite `mode=ro` and `PRAGMA query_only = ON`.


In [ ]:
from __future__ import annotations

import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "pyproject.toml").is_file() else cwd.parent
DB_PATH = REPO_ROOT / "data" / "uni_rag.sqlite"
connection = sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)
connection.execute("PRAGMA query_only = ON")


def read_df(sql: str) -> pd.DataFrame:
    try:
        return pd.read_sql_query(sql, connection)
    except (sqlite3.Error, ValueError):
        return pd.DataFrame()


def parse_json(value: object, fallback: object = None) -> object:
    if not isinstance(value, str) or not value.strip():
        return fallback
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return fallback

## Search-run status and query planning

Empty tables and malformed historical JSON are expected to produce empty frames rather than notebook failures.

In [ ]:
search_runs = read_df(
    "SELECT id, query_type, status, query_plan_json, retrieval_settings_json, started_at, finished_at, error FROM search_runs ORDER BY id"
)
run_status_counts = (
    search_runs.groupby("status", dropna=False).size().rename("count").reset_index()
    if not search_runs.empty
    else pd.DataFrame(columns=["status", "count"])
)
query_type_counts = (
    search_runs.groupby("query_type", dropna=False).size().rename("count").reset_index()
    if not search_runs.empty
    else pd.DataFrame(columns=["query_type", "count"])
)
planning_rows = []
for row in search_runs.itertuples(index=False):
    plan = parse_json(row.query_plan_json, {})
    planning_rows.append(
        {
            "search_run_id": row.id,
            "plan_confidence": plan.get("plan_confidence")
            if isinstance(plan, dict)
            else None,
            "query_type": row.query_type,
            "status": row.status,
        }
    )
planning = pd.DataFrame(planning_rows)
run_status_counts, query_type_counts, planning.head()

In [ ]:
settings_rows = []
for row in search_runs.itertuples(index=False):
    settings = parse_json(row.retrieval_settings_json, {})
    if isinstance(settings, dict):
        settings_rows.append(
            {
                "search_run_id": row.id,
                "embedding_model": settings.get("embedding_model"),
                "llm_provider": settings.get("llm_provider"),
                "final_top_k": settings.get("final_top_k"),
                "evidence_max_tokens": settings.get("evidence_max_tokens"),
                "conversation_context_message_count": settings.get(
                    "conversation_context_message_count"
                ),
            }
        )
settings_snapshots = pd.DataFrame(settings_rows)
if not planning.empty and planning["plan_confidence"].notna().any():
    planning["plan_confidence"].plot(kind="hist", bins=10, title="Planning confidence")
    plt.show()
settings_snapshots

## Raw result sets, complete fusion, and contribution mix

In [ ]:
result_set_envelopes = read_df(
    "SELECT search_run_id, result_set_id, retrieval_method, query, result_count, completed_at FROM search_result_sets ORDER BY id"
)
result_set_counts = (
    result_set_envelopes.groupby(["search_run_id", "retrieval_method"])
    .agg(
        result_set_count=("result_set_id", "size"),
        raw_result_count=("result_count", "sum"),
    )
    .reset_index()
    if not result_set_envelopes.empty
    else pd.DataFrame(
        columns=[
            "search_run_id",
            "retrieval_method",
            "result_set_count",
            "raw_result_count",
        ]
    )
)
raw_results = read_df(
    "SELECT search_run_id, retrieval_method, rank, score, selected_for_evidence, result_json FROM search_results WHERE retrieval_method <> 'hybrid'"
)
fused_results = read_df(
    "SELECT search_run_id, retrieval_method, rank, score, selected_for_evidence, result_json FROM search_results WHERE retrieval_method = 'hybrid'"
)
raw_counts_by_method = (
    raw_results.groupby(["search_run_id", "retrieval_method"])
    .size()
    .rename("count")
    .reset_index()
    if not raw_results.empty
    else pd.DataFrame(columns=["search_run_id", "retrieval_method", "count"])
)
semantic_queries = []
for row in raw_results.itertuples(index=False):
    payload = parse_json(row.result_json, {})
    if isinstance(payload, dict) and row.retrieval_method == "semantic":
        semantic_queries.append(
            {
                "search_run_id": row.search_run_id,
                "result_set_id": payload.get("result_set_id"),
                "query": payload.get("result_set_query"),
            }
        )
semantic_query_rows = pd.DataFrame(semantic_queries)
fused_rank_score = (
    fused_results[["search_run_id", "rank", "score"]]
    if not fused_results.empty
    else pd.DataFrame(columns=["search_run_id", "rank", "score"])
)
if not fused_rank_score.empty:
    fused_rank_score["score"].plot(
        kind="hist", bins=20, title="Complete fused RRF score distribution"
    )
    plt.show()
result_set_counts, raw_counts_by_method, semantic_query_rows, fused_rank_score.head()

In [ ]:
contribution_rows = []
for row in fused_results.itertuples(index=False):
    payload = parse_json(row.result_json, {})
    contributions = (
        payload.get("contributions", []) if isinstance(payload, dict) else []
    )
    for contribution in contributions if isinstance(contributions, list) else []:
        if isinstance(contribution, dict):
            contribution_rows.append(
                {
                    "search_run_id": row.search_run_id,
                    "retrieval_method": contribution.get("retrieval_method"),
                    "result_set_id": contribution.get("result_set_id"),
                    "rrf_contribution": contribution.get("rrf_contribution"),
                }
            )
contribution_mix = pd.DataFrame(contribution_rows)
contribution_mix.groupby("retrieval_method").size().rename(
    "count"
).reset_index() if not contribution_mix.empty else pd.DataFrame(
    columns=["retrieval_method", "count"]
)

## Selected evidence, coverage, weaknesses, and failure diagnostics

In [ ]:
packets = read_df(
    "SELECT id, search_run_id, evidence_count, packet_json, created_at FROM evidence_packets ORDER BY id"
)
coverage_rows = []
weakness_rows = []
for row in packets.itertuples(index=False):
    packet = parse_json(row.packet_json, {})
    coverage = packet.get("coverage", {}) if isinstance(packet, dict) else {}
    if isinstance(coverage, dict):
        coverage_rows.append(
            {
                "search_run_id": row.search_run_id,
                "status": coverage.get("status"),
                "evidence_count": coverage.get("evidence_count"),
                "evidence_token_count": coverage.get("evidence_token_count"),
                "evidence_max_tokens": (packet.get("retrieval_settings") or {}).get(
                    "evidence_max_tokens"
                )
                if isinstance(packet, dict)
                else None,
                "fused_candidate_count": coverage.get("fused_candidate_count"),
                "file_only_candidate_count": coverage.get("file_only_candidate_count"),
                "token_budget_omission_count": coverage.get(
                    "token_budget_omission_count"
                ),
                "oversized_evidence_omission_count": coverage.get(
                    "oversized_evidence_omission_count"
                ),
            }
        )
    weaknesses = packet.get("weaknesses", []) if isinstance(packet, dict) else []
    for weakness in weaknesses if isinstance(weaknesses, list) else []:
        weakness_rows.append({"search_run_id": row.search_run_id, "weakness": weakness})
coverage = pd.DataFrame(coverage_rows)
weaknesses = pd.DataFrame(weakness_rows)
selected_hybrid = (
    fused_results[fused_results["selected_for_evidence"] == 1]
    if not fused_results.empty
    else fused_results
)
rejected_hybrid = (
    fused_results[fused_results["selected_for_evidence"] == 0]
    if not fused_results.empty
    else fused_results
)
coverage, selected_hybrid.head(), rejected_hybrid.head()

In [ ]:
weakness_frequency = (
    weaknesses.groupby("weakness")
    .size()
    .sort_values(ascending=False)
    .rename("count")
    .reset_index()
    if not weaknesses.empty
    else pd.DataFrame(columns=["weakness", "count"])
)
if not weakness_frequency.empty:
    weakness_frequency.head(20).plot(
        kind="barh", x="weakness", y="count", legend=False, title="Weakness frequency"
    )
    plt.show()
missing_scope_rows = []
for row in packets.itertuples(index=False):
    packet = parse_json(row.packet_json, {})
    packet_coverage = packet.get("coverage", {}) if isinstance(packet, dict) else {}
    for field in (
        "courses_without_chunk_hits",
        "indexes_without_chunk_hits",
        "missing_capabilities",
        "semantic_queries_without_hits",
    ):
        values = (
            packet_coverage.get(field, []) if isinstance(packet_coverage, dict) else []
        )
        for value in values if isinstance(values, list) else []:
            missing_scope_rows.append(
                {"search_run_id": row.search_run_id, "kind": field, "value": value}
            )
missing_scope = pd.DataFrame(missing_scope_rows)
weakness_frequency, missing_scope.head()

In [ ]:
failed_runs = (
    search_runs[search_runs["status"] == "failed"]
    if not search_runs.empty
    else search_runs
)
failed_run_ids = (
    tuple(int(value) for value in failed_runs["id"].dropna())
    if not failed_runs.empty
    else ()
)
partial_result_diagnostics = (
    raw_results[raw_results["search_run_id"].isin(failed_run_ids)]
    if not raw_results.empty
    else raw_results
)
unsupported_or_zero_evidence = (
    coverage[coverage["evidence_count"].fillna(0) == 0]
    if not coverage.empty
    else coverage
)
(
    failed_runs[["id", "status", "error"]]
    if not failed_runs.empty
    else pd.DataFrame(columns=["id", "status", "error"]),
    partial_result_diagnostics.head(),
    unsupported_or_zero_evidence,
)

In [ ]:
connection.close()